In [1]:
"""
ACTIVIDAD 1: CONTEXTO SISMOTECTÓNICO Y LEYES EMPÍRICAS
Programa para explicar el contexto sismotectónico
Diploma de sismología 2026

Realizó: Rogelio Torres Cuenca
rogelio.torres@ug.uchile.cl
2026


"""

# =====================================================================
# 0. IMPORTAR LIBRERÍAS
# =====================================================================

from pathlib import Path
import base64
import io
import math
import zipfile

import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit


# =====================================================================
# 1. CONFIGURACIÓN GENERAL
# =====================================================================

ARCHIVO_CATALOGO = "CatalogoSismico.txt"
CARPETA_SALIDA = "Resultados_Actividad1"

# Estilo general de las figuras.
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 13
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

COLOR_MAR = "#DCEEF8"
COLOR_LINEA = "#D62728"
COLOR_CORREDOR = "#F4A6A6"
COLOR_GR = "lightblue"
COLOR_PUNTOS_GR = "darkred"
COLOR_TIERRA = "#E8E0D2"
COLOR_FRONTERA = "#4A4A4A"

# Mapa mundial simplificado de países. Se extrae automáticamente en una
# carpeta interna la primera vez que se ejecuta el programa.
MAPA_PAISES_B64 = """
...
"""


# =====================================================================
# 2. FUNCIONES BÁSICAS
# =====================================================================

def pedir_numero(mensaje, valor_sugerido):
    """Solicita un número y permite aceptar el valor sugerido con Enter."""
    while True:
        texto = input(f"{mensaje} [{valor_sugerido}]: ").strip()

        if texto == "":
            return float(valor_sugerido)

        try:
            return float(texto)
        except ValueError:
            print("Entrada no válida. Escriba un número.")


def pedir_entero(mensaje, valor_sugerido):
    """Solicita un número entero."""
    while True:
        valor = pedir_numero(mensaje, valor_sugerido)

        if float(valor).is_integer():
            return int(valor)

        print("La entrada debe ser un número entero.")


def pedir_si_no(mensaje, valor_sugerido="s"):
    """Solicita una respuesta s/n."""
    while True:
        texto = input(f"{mensaje} (s/n) [{valor_sugerido}]: ").strip().lower()

        if texto == "":
            texto = valor_sugerido

        if texto in ("s", "n"):
            return texto

        print("Responda solamente s o n.")


def pausa(mensaje="Presione Enter para continuar..."):
    """Pausa para permitir la explicación del profesor."""
    input("\n" + mensaje)


def guardar_figura(figura, carpeta, nombre):
    """Guarda una figura con resolución adecuada para el informe."""
    ruta = carpeta / nombre
    figura.savefig(ruta, dpi=300, bbox_inches="tight", facecolor="white")
    print("Figura guardada:", ruta.name)


def mostrar_figura(figura, mensaje=None):
    """
    Muestra la figura en una ventana y espera a que el usuario la cierre.

    Esto permite revisar cada resultado antes de continuar con el siguiente
    paso del análisis.
    """
    if mensaje:
        print("\n" + mensaje)

    figura.tight_layout()
    plt.show(block=True)


def omori(t, k, c, p):
    """Ley de Omori modificada."""
    return k / (t + c) ** p

def backend_es_interactivo():
    """
    Comprueba si Matplotlib está usando un backend que abre ventanas.

    QtAgg y TkAgg son interactivos, aunque sus nombres contienen "Agg".
    """
    backend = plt.get_backend().lower()

    backends_no_interactivos = (
        "agg",
        "pdf",
        "ps",
        "svg",
        "template",
        "cairo",
        "module://matplotlib_inline.backend_inline",
    )

    if "inline" in backend:
        return False

    return backend not in backends_no_interactivos


def activar_backend_interactivo():
    """
    Intenta activar una ventana gráfica externa.

    En Spyder primero intenta ejecutar el equivalente de:
        %matplotlib qt

    Si no es posible, intenta cambiar a QtAgg o TkAgg.
    """
    if backend_es_interactivo():
        print("Backend gráfico activo:", plt.get_backend())
        return True

    try:
        ipython = get_ipython()
        ipython.run_line_magic("matplotlib", "qt")
        plt.pause(0.2)

        if backend_es_interactivo():
            print("Backend gráfico activado:", plt.get_backend())
            return True
    except Exception:
        pass

    for backend in ("QtAgg", "TkAgg"):
        try:
            plt.switch_backend(backend)
            plt.pause(0.2)

            if backend_es_interactivo():
                print("Backend gráfico activado:", plt.get_backend())
                return True
        except Exception:
            continue

    print(
        "\nADVERTENCIA: no fue posible activar una ventana gráfica externa.\n"
        "En Spyder cambie:\n"
        "Herramientas > Preferencias > Consola IPython > Gráficos > Qt\n"
        "y reinicie el núcleo."
    )

    return False


def seleccionar_dos_puntos_en_figura(figura, eje):
    """
    Registra dos clics izquierdos en el eje.

    Primer clic  -> A
    Segundo clic -> B

    Después del segundo clic, la línea A-B queda dibujada.
    El usuario debe cerrar la ventana para continuar.
    """
    seleccion = []

    def al_hacer_clic(evento):
        if evento.inaxes is not eje:
            return

        if evento.button != 1:
            return

        if evento.xdata is None or evento.ydata is None:
            return

        if len(seleccion) >= 2:
            return

        seleccion.append((evento.xdata, evento.ydata))

        etiqueta = "A" if len(seleccion) == 1 else "B"

        eje.scatter(
            evento.xdata,
            evento.ydata,
            s=110,
            color=COLOR_LINEA,
            edgecolors="white",
            linewidths=1.5,
            zorder=20,
        )

        eje.text(
            evento.xdata,
            evento.ydata,
            " " + etiqueta,
            color=COLOR_LINEA,
            fontweight="bold",
            fontsize=14,
            zorder=21,
        )

        if len(seleccion) == 1:
            eje.set_title(
                "Punto A registrado\n"
                "Haga clic en el punto B"
            )
        else:
            eje.plot(
                [seleccion[0][0], seleccion[1][0]],
                [seleccion[0][1], seleccion[1][1]],
                color=COLOR_LINEA,
                linewidth=3,
                zorder=19,
            )

            eje.set_title(
                "Sección A-B registrada\n"
                "Revise la línea y cierre esta ventana para continuar"
            )

        figura.canvas.draw_idle()

    identificador = figura.canvas.mpl_connect(
        "button_press_event",
        al_hacer_clic,
    )

    plt.show(block=True)

    figura.canvas.mpl_disconnect(identificador)

    return seleccion


def seleccionar_valor_x_en_figura(figura, eje, delta_x):
    """
    Permite elegir un valor de X con un clic izquierdo.

    Se usa para seleccionar Mc sobre Gutenberg-Richter.
    Después del clic, el usuario debe cerrar la ventana.
    """
    seleccion = []

    def al_hacer_clic(evento):
        if evento.inaxes is not eje:
            return

        if evento.button != 1:
            return

        if evento.xdata is None:
            return

        if len(seleccion) >= 1:
            return

        valor = round(evento.xdata / delta_x) * delta_x
        seleccion.append(valor)

        eje.axvline(
            valor,
            color="crimson",
            linestyle="--",
            linewidth=2.2,
            zorder=20,
        )

        eje.set_title(
            f"Mc seleccionada = {valor:.2f}\n"
            "Revise la selección y cierre esta ventana para continuar"
        )

        figura.canvas.draw_idle()

    identificador = figura.canvas.mpl_connect(
        "button_press_event",
        al_hacer_clic,
    )

    plt.show(block=True)

    figura.canvas.mpl_disconnect(identificador)

    return seleccion


# =====================================================================
# 3. LEER LOS ARCHIVOS
# =====================================================================

def leer_catalogo(ruta):
    """
    Lee el catálogo sísmico con el formato:

        time    latitude    longitude    depth    mag

    La primera fila debe contener esos cinco encabezados.
    Las columnas pueden estar separadas por tabulaciones o espacios.
    """
    datos = pd.read_csv(
        ruta,
        sep=',',
        engine="python",
    )

    # Normalizar los encabezados para evitar problemas por mayúsculas,
    # espacios o caracteres invisibles.
    datos.columns = [
        str(columna).strip().lower().lstrip("\ufeff")
        for columna in datos.columns
    ]

    columnas_requeridas = [
        "time",
        "latitude",
        "longitude",
        "depth",
        "mag",
    ]

    columnas_faltantes = [
        columna
        for columna in columnas_requeridas
        if columna not in datos.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            "El catálogo no contiene el formato requerido.\n"
            "Las columnas deben ser:\n"
            "time  latitude  longitude  depth  mag\n\n"
            "Columnas faltantes: "
            + ", ".join(columnas_faltantes)
        )

    numero_original = len(datos)

    catalogo = pd.DataFrame()

    # Las fechas terminadas en Z se interpretan en UTC y posteriormente
    # se convierten a fechas sin zona horaria para facilitar las gráficas.
    catalogo["Fecha"] = pd.to_datetime(
        datos["time"],
        errors="coerce",
        utc=True,
    ).dt.tz_convert(None)

    catalogo["Latitud"] = pd.to_numeric(
        datos["latitude"],
        errors="coerce",
    )

    catalogo["Longitud"] = pd.to_numeric(
        datos["longitude"],
        errors="coerce",
    )

    catalogo["Profundidad"] = pd.to_numeric(
        datos["depth"],
        errors="coerce",
    )

    catalogo["Magnitud"] = pd.to_numeric(
        datos["mag"],
        errors="coerce",
    )

    # Eliminar únicamente filas incompletas o no numéricas.
    catalogo = catalogo.dropna().reset_index(drop=True)

    numero_eliminado = numero_original - len(catalogo)

    if numero_eliminado > 0:
        print(
            "Advertencia: se eliminaron",
            numero_eliminado,
            "filas con datos incompletos o no válidos.",
        )

    if len(catalogo) == 0:
        raise ValueError(
            "El catálogo no contiene eventos sísmicos válidos."
        )

    # Crear las columnas separadas que se utilizan en algunas etapas
    # del análisis y en los archivos de salida.
    catalogo["Año"] = catalogo["Fecha"].dt.year
    catalogo["Mes"] = catalogo["Fecha"].dt.month
    catalogo["Día"] = catalogo["Fecha"].dt.day
    catalogo["Hora"] = catalogo["Fecha"].dt.hour
    catalogo["Minuto"] = catalogo["Fecha"].dt.minute

    catalogo["Segundo"] = (
        catalogo["Fecha"].dt.second
        + catalogo["Fecha"].dt.microsecond / 1_000_000
    )

    catalogo = catalogo[
        [
            "Año",
            "Mes",
            "Día",
            "Hora",
            "Minuto",
            "Segundo",
            "Latitud",
            "Longitud",
            "Profundidad",
            "Magnitud",
            "Fecha",
        ]
    ]

    catalogo = catalogo.sort_values(
        "Fecha"
    ).reset_index(drop=True)

    return catalogo


def cargar_mapa_paises(carpeta_programa):
    """
    Extrae y carga el mapa mundial incluido dentro del programa.

    La extracción se realiza solamente la primera vez. Después se reutiliza
    la carpeta creada junto al código.
    """
    carpeta_mapa = carpeta_programa / "_mapa_base_paises"
    archivo_shp = carpeta_mapa / "naturalearth_lowres.shp"

    if not archivo_shp.exists():
        carpeta_mapa.mkdir(parents=True, exist_ok=True)

        datos_zip = base64.b64decode(
            "".join(MAPA_PAISES_B64.split())
        )

        with zipfile.ZipFile(io.BytesIO(datos_zip), "r") as archivo_zip:
            archivo_zip.extractall(carpeta_mapa)

    paises = gpd.read_file(archivo_shp)

    if paises.crs is None:
        paises = paises.set_crs("EPSG:4326")
    else:
        paises = paises.to_crs("EPSG:4326")

    return paises


# =====================================================================
# 4. FUNCIONES PARA EL MAPA
# =====================================================================

def corregir_aspecto_mapa(eje, latitud_media):
    """Corrige aproximadamente la deformación longitud-latitud."""
    coseno = math.cos(math.radians(latitud_media))

    if abs(coseno) > 0.01:
        eje.set_aspect(1.0 / coseno)


def limites_sugeridos(catalogo):
    """Calcula límites iniciales a partir del catálogo."""
    rango_lon = catalogo["Longitud"].max() - catalogo["Longitud"].min()
    rango_lat = catalogo["Latitud"].max() - catalogo["Latitud"].min()

    margen_lon = max(0.25, 0.05 * rango_lon)
    margen_lat = max(0.25, 0.05 * rango_lat)

    return (
        catalogo["Longitud"].min() - margen_lon,
        catalogo["Longitud"].max() + margen_lon,
        catalogo["Latitud"].min() - margen_lat,
        catalogo["Latitud"].max() + margen_lat,
    )


def dibujar_mapa(eje, catalogo, paises, limites):
    """Dibuja países, continente, océano y sismicidad."""
    lon_min, lon_max, lat_min, lat_max = limites

    eje.set_facecolor(COLOR_MAR)

    # Recortar visualmente el mapa mundial al entorno de la región.
    margen_lon = max(1.0, 0.10 * (lon_max - lon_min))
    margen_lat = max(1.0, 0.10 * (lat_max - lat_min))

    paises_visibles = paises.cx[
        lon_min - margen_lon : lon_max + margen_lon,
        lat_min - margen_lat : lat_max + margen_lat,
    ]

    if len(paises_visibles) > 0:
        paises_visibles.plot(
            ax=eje,
            facecolor=COLOR_TIERRA,
            edgecolor=COLOR_FRONTERA,
            linewidth=0.65,
            zorder=1,
        )

    magnitud_minima = catalogo["Magnitud"].min()
    tamanos = 14 + 10 * (catalogo["Magnitud"] - magnitud_minima)

    puntos = eje.scatter(
        catalogo["Longitud"],
        catalogo["Latitud"],
        c=catalogo["Profundidad"],
        s=tamanos,
        cmap="plasma_r",
        alpha=0.82,
        edgecolors="black",
        linewidths=0.25,
        label="Sismos",
        zorder=4,
    )

    eje.set_xlim(lon_min, lon_max)
    eje.set_ylim(lat_min, lat_max)
    corregir_aspecto_mapa(eje, 0.5 * (lat_min + lat_max))

    eje.set_xlabel("Longitud [°]")
    eje.set_ylabel("Latitud [°]")
    eje.grid(alpha=0.25, linestyle=":")

    return puntos


def recortar_catalogo(catalogo, limites):
    """Selecciona eventos dentro de un rectángulo longitud-latitud."""
    lon_min, lon_max, lat_min, lat_max = limites

    mascara = (
        (catalogo["Longitud"] >= lon_min)
        & (catalogo["Longitud"] <= lon_max)
        & (catalogo["Latitud"] >= lat_min)
        & (catalogo["Latitud"] <= lat_max)
    )

    return catalogo.loc[mascara].copy().reset_index(drop=True)


def seleccionar_puntos_ab(catalogo, paises, limites):
    """
    Abre una ventana externa para definir una sección A-B.

    Primer clic izquierdo  = punto A
    Segundo clic izquierdo = punto B

    Después del segundo clic, la línea queda dibujada.
    Cierre la ventana para regresar a la consola.
    """
    while True:
        figura, eje = plt.subplots(figsize=(10, 8))

        puntos = dibujar_mapa(
            eje,
            catalogo,
            paises,
            limites,
        )

        eje.set_title(
            "Definición de la sección A-B\n"
            "Haga clic primero en A"
        )

        eje.legend(loc="best")

        barra = figura.colorbar(
            puntos,
            ax=eje,
            pad=0.02,
        )
        barra.set_label("Profundidad [km]")

        figura.tight_layout()

        if not backend_es_interactivo():
            plt.close(figura)

            raise RuntimeError(
                "Matplotlib no está usando una ventana interactiva.\n\n"
                "En Spyder seleccione:\n"
                "Herramientas > Preferencias > Consola IPython > "
                "Gráficos > Backend: Qt\n\n"
                "Después reinicie el núcleo y ejecute nuevamente."
            )

        print("\nSELECCIÓN GRÁFICA DE LA SECCIÓN")
        print("1. Se abrirá una ventana independiente.")
        print("2. Haga clic izquierdo en el punto A.")
        print("3. Haga clic izquierdo en el punto B.")
        print("4. Revise la línea y cierre la ventana para continuar.")

        seleccion = seleccionar_dos_puntos_en_figura(
            figura,
            eje,
        )

        plt.close(figura)

        if len(seleccion) != 2:
            print(
                "\nLa ventana se cerró antes de registrar A y B."
            )

            repetir = pedir_si_no(
                "¿Desea abrir nuevamente el mapa?",
                "s",
            )

            if repetir == "s":
                continue

            raise RuntimeError(
                "No se completó la selección de la sección A-B."
            )

        lon_a, lat_a = seleccion[0]
        lon_b, lat_b = seleccion[1]

        if math.isclose(lon_a, lon_b) and math.isclose(lat_a, lat_b):
            print("A y B no pueden estar en el mismo punto.")
            continue

        print("\nCoordenadas seleccionadas")
        print(
            "A = (Longitud %.5f°, Latitud %.5f°)"
            % (lon_a, lat_a)
        )
        print(
            "B = (Longitud %.5f°, Latitud %.5f°)"
            % (lon_b, lat_b)
        )

        aceptar = pedir_si_no(
            "¿Acepta la posición de A y B?",
            "s",
        )

        if aceptar == "s":
            return lon_a, lat_a, lon_b, lat_b

        print("Se abrirá nuevamente el mapa para repetir la selección.")


# =====================================================================
# 5. SELECCIONAR LOS EVENTOS DEL PERFIL
# =====================================================================

def calcular_perfil(catalogo, lon_a, lat_a, lon_b, lat_b, semiancho_grados):
    """
    Selecciona eventos dentro de un corredor paralelo a A-B.

    semiancho_grados = 1 significa una franja de ±1° alrededor de la línea.
    """
    latitud_media = 0.5 * (lat_a + lat_b)

    km_por_grado_lat = 111.2
    km_por_grado_lon = 111.2 * math.cos(math.radians(latitud_media))

    # Coordenadas de los eventos respecto de A, expresadas en kilómetros.
    x = (catalogo["Longitud"].to_numpy() - lon_a) * km_por_grado_lon
    y = (catalogo["Latitud"].to_numpy() - lat_a) * km_por_grado_lat

    # Vector A-B en kilómetros.
    x_b = (lon_b - lon_a) * km_por_grado_lon
    y_b = (lat_b - lat_a) * km_por_grado_lat
    longitud_ab = math.sqrt(x_b**2 + y_b**2)

    if longitud_ab == 0:
        raise ValueError("La longitud de la sección A-B es cero.")

    # t indica la posición proyectada del evento a lo largo de A-B.
    t = (x * x_b + y * y_b) / longitud_ab**2
    distancia_perfil = t * longitud_ab

    # Distancia perpendicular de cada evento a la línea A-B.
    distancia_perpendicular = np.abs(x * y_b - y * x_b) / longitud_ab

    semiancho_km = semiancho_grados * 111.2

    mascara = (
        (t >= 0)
        & (t <= 1)
        & (distancia_perpendicular <= semiancho_km)
    )

    perfil = catalogo.loc[mascara].copy().reset_index(drop=True)
    perfil["Distancia_perfil_km"] = distancia_perfil[mascara]
    perfil["Distancia_perpendicular_km"] = distancia_perpendicular[mascara]

    # Bordes del corredor para dibujarlos en el mapa.
    normal_x = -y_b / longitud_ab
    normal_y = x_b / longitud_ab
    desplazamiento_x = semiancho_km * normal_x
    desplazamiento_y = semiancho_km * normal_y

    borde_1 = [
        (
            lon_a + desplazamiento_x / km_por_grado_lon,
            lat_a + desplazamiento_y / km_por_grado_lat,
        ),
        (
            lon_b + desplazamiento_x / km_por_grado_lon,
            lat_b + desplazamiento_y / km_por_grado_lat,
        ),
    ]

    borde_2 = [
        (
            lon_a - desplazamiento_x / km_por_grado_lon,
            lat_a - desplazamiento_y / km_por_grado_lat,
        ),
        (
            lon_b - desplazamiento_x / km_por_grado_lon,
            lat_b - desplazamiento_y / km_por_grado_lat,
        ),
    ]

    return perfil, longitud_ab, borde_1, borde_2


# =====================================================================
# 6. PROGRAMA PRINCIPAL
# =====================================================================

def main():
    # Forzar una ventana externa antes de crear cualquier figura.
    if not activar_backend_interactivo():
        raise RuntimeError(
            "Se requiere un backend gráfico interactivo para seleccionar "
            "A y B directamente sobre el mapa."
        )

    carpeta_programa = Path('/home/keneth/Documents/Academic_Projects/AGCID2026_DIS_UChile/Week1/Task_files')

    ruta_catalogo = carpeta_programa / ARCHIVO_CATALOGO
    carpeta_salida = carpeta_programa / CARPETA_SALIDA
    carpeta_salida.mkdir(parents=True, exist_ok=True)

    if not ruta_catalogo.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo:\n{ruta_catalogo}\n\n"
            "Coloque el programa y el catálogo en la misma carpeta."
        )

    catalogo = leer_catalogo(ruta_catalogo)
    paises = cargar_mapa_paises(carpeta_programa)

    print("\n" + "=" * 78)
    print("ACTIVIDAD 1: CONTEXTO SISMOTECTÓNICO Y LEYES EMPÍRICAS")
    print("PROGRAMA DOCENTE BASADO EN EL NOTEBOOK ORIGINAL")
    print("=" * 78)

    print("\nRESUMEN DEL CATÁLOGO")
    print("Número de eventos:", len(catalogo))
    print("Fecha inicial:", catalogo["Fecha"].min())
    print("Fecha final:", catalogo["Fecha"].max())
    print(
        "Latitud: %.3f a %.3f°"
        % (catalogo["Latitud"].min(), catalogo["Latitud"].max())
    )
    print(
        "Longitud: %.3f a %.3f°"
        % (catalogo["Longitud"].min(), catalogo["Longitud"].max())
    )
    print(
        "Profundidad: %.1f a %.1f km"
        % (catalogo["Profundidad"].min(), catalogo["Profundidad"].max())
    )
    print(
        "Magnitud: %.2f a %.2f"
        % (catalogo["Magnitud"].min(), catalogo["Magnitud"].max())
    )

    catalogo.to_csv(carpeta_salida / "Catalogo_procesado.csv", index=False)

    pausa(
        f"Se cargó el archivo de datos: {ruta_catalogo.name}. "
        "Presione Enter para mostrar la sismicidad en planta..."
    )

    # -----------------------------------------------------------------
    # PASO 1. SISMICIDAD GENERAL EN PLANTA
    # -----------------------------------------------------------------
    print("\n" + "-" * 78)
    print("PASO 1. SISMICIDAD EN PLANTA")
    print("-" * 78)

    limites_iniciales = limites_sugeridos(catalogo)

    figura, eje = plt.subplots(figsize=(10, 8))
    puntos = dibujar_mapa(eje, catalogo, paises, limites_iniciales)
    eje.set_title("Sismicidad en planta")
    eje.legend(loc="best")

    barra = figura.colorbar(puntos, ax=eje, pad=0.02)
    barra.set_label("Profundidad [km]")

    figura.tight_layout()
    guardar_figura(figura, carpeta_salida, "01_Sismicidad_general_en_planta.png")
    mostrar_figura(figura, "Se muestra la figura general de sismicidad en planta. Revísela y cierre la ventana para continuar.")
    plt.close(figura)


    # -----------------------------------------------------------------
    # PASO 2. ACOTAR LA REGIÓN
    # -----------------------------------------------------------------
    print("\n" + "-" * 78)
    print("PASO 2. DEFINIR LOS LÍMITES DEL ÁREA DE ANÁLISIS")
    print("-" * 78)

    lon_min_s, lon_max_s, lat_min_s, lat_max_s = limites_iniciales

    lon_min = pedir_numero("Longitud mínima", round(lon_min_s, 2))
    lon_max = pedir_numero("Longitud máxima", round(lon_max_s, 2))
    lat_min = pedir_numero("Latitud mínima", round(lat_min_s, 2))
    lat_max = pedir_numero("Latitud máxima", round(lat_max_s, 2))

    if lon_min >= lon_max or lat_min >= lat_max:
        raise ValueError("Los límites mínimos deben ser menores que los máximos.")

    limites_region = (lon_min, lon_max, lat_min, lat_max)
    region = recortar_catalogo(catalogo, limites_region)

    if len(region) < 10:
        raise ValueError(
            "La región seleccionada contiene menos de diez eventos. "
            "Amplíe los límites."
        )

    print("Eventos incluidos en la región:", len(region))

    figura, eje = plt.subplots(figsize=(10, 8))
    puntos = dibujar_mapa(eje, region, paises, limites_region)
    eje.set_title("Región seleccionada para el análisis")
    eje.legend(loc="best")

    barra = figura.colorbar(puntos, ax=eje, pad=0.02)
    barra.set_label("Profundidad [km]")

    figura.tight_layout()
    guardar_figura(figura, carpeta_salida, "02_Region_seleccionada.png")
    mostrar_figura(figura, "Se muestra la región seleccionada. Revísela y cierre la ventana para continuar.")
    plt.close(figura)

    region.to_csv(carpeta_salida / "Catalogo_region_seleccionada.csv", index=False)

    pausa("Continuar...")

    # -----------------------------------------------------------------
    # PASO 3. TRAZAR A-B Y CONSTRUIR EL PERFIL
    # -----------------------------------------------------------------
    print("\n" + "-" * 78)
    print("PASO 3. TRAZAR DIRECTAMENTE LA SECCIÓN A-B")
    print("-" * 78)

    while True:
        lon_a, lat_a, lon_b, lat_b = seleccionar_puntos_ab(
            region, paises, limites_region
        )

        # El valor solicitado es ±1°, pero queda editable para la clase.
        semiancho = pedir_numero(
            "Semiancho del corredor alrededor de A-B, en grados", 1.0
        )

        if semiancho <= 0:
            print("El semiancho debe ser mayor que cero.")
            continue

        perfil, longitud_ab, borde_1, borde_2 = calcular_perfil(
            region, lon_a, lat_a, lon_b, lat_b, semiancho
        )

        print("Longitud aproximada de A-B: %.1f km" % longitud_ab)
        print("Eventos dentro del corredor:", len(perfil))

        if len(perfil) == 0:
            print("No hay eventos dentro del corredor. Trace otra línea o amplíelo.")
            continue

        figura, eje = plt.subplots(figsize=(10, 8))
        puntos = dibujar_mapa(eje, region, paises, limites_region)

        eje.fill(
            [borde_1[0][0], borde_1[1][0], borde_2[1][0], borde_2[0][0]],
            [borde_1[0][1], borde_1[1][1], borde_2[1][1], borde_2[0][1]],
            color=COLOR_CORREDOR,
            alpha=0.30,
            label=f"Corredor ±{semiancho:.2f}°",
            zorder=2,
        )

        eje.plot(
            [lon_a, lon_b],
            [lat_a, lat_b],
            color=COLOR_LINEA,
            linewidth=3,
            label="Sección A-B",
            zorder=7,
        )

        eje.scatter(
            perfil["Longitud"],
            perfil["Latitud"],
            s=38,
            facecolors="none",
            edgecolors="cyan",
            linewidths=1.3,
            label="Eventos usados en el perfil",
            zorder=8,
        )

        eje.scatter(
            [lon_a, lon_b],
            [lat_a, lat_b],
            s=95,
            color=COLOR_LINEA,
            edgecolors="white",
            linewidths=1.4,
            zorder=9,
        )

        eje.text(lon_a, lat_a, " A", color=COLOR_LINEA, fontweight="bold", fontsize=13)
        eje.text(lon_b, lat_b, " B", color=COLOR_LINEA, fontweight="bold", fontsize=13)
        eje.set_title("Sección A-B y corredor utilizado para el perfil")
        eje.legend(loc="best")

        barra = figura.colorbar(puntos, ax=eje, pad=0.02)
        barra.set_label("Profundidad [km]")

        figura.tight_layout()
        guardar_figura(figura, carpeta_salida, "03_Seccion_AB_y_corredor.png")
        mostrar_figura(figura, "Se muestra la sección A-B con el corredor. Revísela y cierre la ventana para decidir si la acepta.")
        plt.close(figura)

        aceptar = pedir_si_no("¿Acepta esta sección y el corredor?", "s")

        if aceptar == "s":
            break

    perfil.to_csv(carpeta_salida / "Catalogo_perfil_AB.csv", index=False)

    profundidad_sugerida = math.ceil(perfil["Profundidad"].max() / 10) * 10
    profundidad_maxima = pedir_numero(
        "Profundidad máxima mostrada en el perfil", profundidad_sugerida
    )

    figura, eje = plt.subplots(figsize=(12, 6))

    tamanos = 17 + 10 * (perfil["Magnitud"] - perfil["Magnitud"].min())

    puntos = eje.scatter(
        perfil["Distancia_perfil_km"],
        perfil["Profundidad"],
        c=perfil["Magnitud"],
        s=tamanos,
        cmap="viridis",
        alpha=0.82,
        edgecolors="black",
        linewidths=0.3,
    )

    eje.set_xlim(0, longitud_ab)
    eje.set_ylim(profundidad_maxima, 0)
    eje.set_xlabel("Distancia desde A a lo largo del perfil [km]")
    eje.set_ylabel("Profundidad [km]")
    eje.set_title("Perfil transversal de sismicidad A-B")
    eje.grid(alpha=0.25, linestyle=":")

    eje.text(
        0,
        -0.07,
        "A",
        transform=eje.get_xaxis_transform(),
        color=COLOR_LINEA,
        fontweight="bold",
        fontsize=13,
        ha="center",
    )

    eje.text(
        longitud_ab,
        -0.07,
        "B",
        transform=eje.get_xaxis_transform(),
        color=COLOR_LINEA,
        fontweight="bold",
        fontsize=13,
        ha="center",
    )

    barra = figura.colorbar(puntos, ax=eje, pad=0.02)
    barra.set_label("Magnitud Mw")

    figura.tight_layout()
    guardar_figura(figura, carpeta_salida, "04_Perfil_transversal_AB.png")
    mostrar_figura(figura, "Se muestra el perfil transversal A-B. Revíselo y cierre la ventana para continuar.")
    plt.close(figura)


    pausa("Continuar...")

    # -----------------------------------------------------------------
    # PASO 4. PERFIL EN TIEMPO COMO EN EL NOTEBOOK ORIGINAL
    # -----------------------------------------------------------------
    print("\n" + "-" * 78)
    print("PASO 4. LATITUD VERSUS TIEMPO CON HISTOGRAMAS MARGINALES")
    print("-" * 78)

    año_inicial = pedir_entero(
        "Año inicial", int(region["Fecha"].dt.year.min())
    )
    año_final = pedir_entero(
        "Año final", int(region["Fecha"].dt.year.max())
    )

    if año_inicial > año_final:
        raise ValueError("El año inicial no puede ser mayor que el año final.")

    datos_tiempo = region[
        (region["Fecha"].dt.year >= año_inicial)
        & (region["Fecha"].dt.year <= año_final)
    ].copy()

    if len(datos_tiempo) == 0:
        raise ValueError("No existen eventos en el intervalo temporal elegido.")

    bins_tiempo = pedir_entero(
        "Número de intervalos del histograma temporal", 30
    )
    bins_latitud = pedir_entero(
        "Número de intervalos del histograma de latitud", 20
    )

    figura = plt.figure(figsize=(13, 7))
    gs = gridspec.GridSpec(
        2,
        2,
        width_ratios=[4, 1],
        height_ratios=[1, 4],
        hspace=0.05,
        wspace=0.05,
    )

    # Histograma superior: número de eventos a través del tiempo.
    eje_superior = figura.add_subplot(gs[0, 0])
    eje_superior.hist(
        datos_tiempo["Fecha"],
        bins=bins_tiempo,
        color="lightgray",
        edgecolor="black",
    )
    eje_superior.set_ylabel("Sismicidad")
    eje_superior.set_title("Perfil en tiempo")
    eje_superior.tick_params(labelbottom=False)
    eje_superior.grid(alpha=0.15, linestyle=":", axis="y")

    # Histograma derecho: distribución de la sismicidad por latitud.
    eje_derecho = figura.add_subplot(gs[1, 1])
    eje_derecho.hist(
        datos_tiempo["Latitud"],
        bins=bins_latitud,
        orientation="horizontal",
        color="lightgray",
        edgecolor="black",
    )
    eje_derecho.set_xlabel("Sismos acumulados")
    eje_derecho.tick_params(labelleft=False)
    eje_derecho.grid(alpha=0.15, linestyle=":", axis="x")

    # Panel principal: latitud contra tiempo, coloreado por profundidad.
    eje_principal = figura.add_subplot(gs[1, 0], sharex=eje_superior)
    tamanos_tiempo = 18 + 6 * (
        datos_tiempo["Magnitud"] - datos_tiempo["Magnitud"].min()
    )
    puntos_tiempo = eje_principal.scatter(
        datos_tiempo["Fecha"],
        datos_tiempo["Latitud"],
        c=datos_tiempo["Profundidad"],
        cmap="viridis_r",
        s=tamanos_tiempo,
        alpha=0.62,
        edgecolors="#1F77B4",
        linewidths=0.3,
    )
    eje_principal.set_ylabel("Latitud [°]")
    eje_principal.set_xlabel("Tiempo [años]")
    eje_principal.grid(alpha=0.20, linestyle=":")
    eje_principal.xaxis.set_major_locator(mdates.YearLocator())
    eje_principal.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # Reservar un margen a la derecha para que la barra de color no se
    # superponga con el histograma lateral.
    figura.autofmt_xdate()
    figura.subplots_adjust(
        hspace=0.05,
        wspace=0.05,
        right=0.88,
    )

    # Posición manual de la barra de color:
    # [posición horizontal, posición vertical, ancho, altura].
    eje_color = figura.add_axes([0.92, 0.16, 0.018, 0.70])

    barra_tiempo = figura.colorbar(
        puntos_tiempo,
        cax=eje_color,
    )
    barra_tiempo.set_label("Profundidad [km]")

    guardar_figura(figura, carpeta_salida, "05_Sismicidad_en_el_tiempo.png")
    print(
        "\nSe muestra la figura latitud-tiempo con histograma superior "
        "y componente vertical por latitud a la derecha. "
        "Revísela y cierre la ventana para continuar."
    )
    plt.show(block=True)
    plt.close(figura)


    pausa("Continuar...")

    # -----------------------------------------------------------------
    # PASO 5. GUTENBERG-RICHTER: HISTOGRAMA ACUMULADO
    # -----------------------------------------------------------------
    print("\n" + "-" * 78)
    print("PASO 5. LEY DE GUTENBERG-RICHTER")
    print("-" * 78)

    delta_m = pedir_numero("Intervalo de magnitud ΔM", 0.1)

    if delta_m <= 0:
        raise ValueError("El intervalo de magnitud debe ser mayor que cero.")

    magnitudes = region["Magnitud"].to_numpy()

    magnitud_minima = math.floor(magnitudes.min() / delta_m) * delta_m
    magnitud_maxima = math.ceil(magnitudes.max() / delta_m) * delta_m

    bordes = np.arange(
        magnitud_minima,
        magnitud_maxima + delta_m * 1.01,
        delta_m,
    )

    histograma, bordes = np.histogram(magnitudes, bins=bordes)
    numero_acumulado = np.cumsum(histograma[::-1])[::-1]
    magnitudes_gr = bordes[:-1]

    # El último intervalo puede quedar vacío por redondeo. Para la figura
    # logarítmica se muestran solamente valores acumulados mayores que cero.
    mascara_positiva = numero_acumulado > 0
    magnitudes_grafica = magnitudes_gr[mascara_positiva]
    acumulado_grafica = numero_acumulado[mascara_positiva]

    # Figura inicial para elegir Mc.
    figura, eje = plt.subplots(figsize=(11, 6.5))

    eje.bar(
        magnitudes_grafica,
        acumulado_grafica,
        width=delta_m,
        align="edge",
        color=COLOR_GR,
        edgecolor="black",
        linewidth=0.7,
        label="Acumulado",
    )

    eje.plot(
        magnitudes_grafica,
        acumulado_grafica,
        "o",
        color=COLOR_PUNTOS_GR,
        markersize=6,
    )

    eje.set_yscale("log")
    eje.set_xlabel("Magnitud Mw")
    eje.set_ylabel("Número acumulado de eventos")
    eje.set_title(
        "Ley Gutenberg-Richter\n"
        "Seleccione la magnitud de corte Mc sobre la figura"
    )
    eje.set_ylim(1, max(acumulado_grafica) * 8)
    eje.grid(False, which="both", axis="y")
    eje.legend(loc="upper right")
    figura.tight_layout()

    print("\nSe mostrará la figura de Gutenberg-Richter para elegir Mc directamente sobre la gráfica.")

    # Elegir Mc directamente con un clic en una ventana externa.
    print("\nSELECCIÓN GRÁFICA DE Mc")
    print("1. Haga clic izquierdo sobre la magnitud que desea usar como Mc.")
    print("2. La línea vertical mostrará el valor seleccionado.")
    print("3. Cierre la ventana para continuar.")

    seleccion_mc = seleccionar_valor_x_en_figura(
        figura,
        eje,
        delta_m,
    )

    plt.close(figura)

    if len(seleccion_mc) == 1:
        mc = seleccion_mc[0]
        print("Mc seleccionada en la figura: %.2f" % mc)
    else:
        print(
            "La figura se cerró sin seleccionar Mc. "
            "El valor se solicitará por teclado."
        )
        mc = pedir_numero("Magnitud de corte Mc", 5.0)

    magnitud_maxima_ajuste = pedir_numero(
        "Magnitud máxima utilizada en el ajuste lineal",
        round(float(magnitudes_grafica.max()), 2),
    )

    while True:
        mascara_ajuste = (
            (magnitudes_gr >= mc)
            & (magnitudes_gr <= magnitud_maxima_ajuste)
            & (numero_acumulado > 0)
        )
        x_ajuste = magnitudes_gr[mascara_ajuste]
        n_ajuste = numero_acumulado[mascara_ajuste]

        if magnitud_maxima_ajuste <= mc:
            print("La magnitud máxima debe ser mayor que Mc.")
            magnitud_maxima_ajuste = pedir_numero(
                "Nueva magnitud máxima utilizada en el ajuste lineal",
                round(float(magnitudes_grafica.max()), 2),
            )
            continue

        if len(x_ajuste) < 3:
            print("El intervalo seleccionado deja menos de tres puntos para el ajuste.")
            mc = pedir_numero("Nueva magnitud de corte Mc", mc)
            magnitud_maxima_ajuste = pedir_numero(
                "Nueva magnitud máxima utilizada en el ajuste lineal",
                magnitud_maxima_ajuste,
            )
            continue

        log_n = np.log10(n_ajuste)
        coeficientes = np.polyfit(x_ajuste, log_n, 1)
        b = -coeficientes[0]
        a = coeficientes[1]
        modelo_n = 10 ** np.polyval(coeficientes, x_ajuste)

        figura, eje = plt.subplots(figsize=(11, 6.5))

        eje.bar(
            magnitudes_grafica,
            acumulado_grafica,
            width=delta_m,
            align="edge",
            color=COLOR_GR,
            edgecolor="black",
            linewidth=0.7,
            label="Acumulado",
        )

        eje.plot(
            magnitudes_grafica,
            acumulado_grafica,
            "o",
            color=COLOR_PUNTOS_GR,
            markersize=6,
        )

        eje.plot(
            x_ajuste,
            modelo_n,
            "k--",
            linewidth=4,
            label="Ajuste lineal",
        )

        eje.axvline(
            mc,
            color="crimson",
            linestyle=":",
            linewidth=2,
            label=f"Mc = {mc:.2f}",
        )

        eje.axvline(
            magnitud_maxima_ajuste,
            color="navy",
            linestyle=":",
            linewidth=2,
            label=f"M máx. ajuste = {magnitud_maxima_ajuste:.2f}",
        )

        eje.set_yscale("log")
        eje.set_xlabel("Magnitud Mw")
        eje.set_ylabel("Número acumulado de eventos")
        eje.set_title("Ley Gutenberg-Richter")
        eje.set_ylim(1, max(acumulado_grafica) * 8)
        eje.grid(False, which="both", axis="y")

        texto_gr = (
            "Ley de Gutenberg-Richter\n"
            "log₁₀(N) = a - bM\n"
            f"a = {a:.2f}\n"
            f"b = {b:.2f}\n"
            f"Mc = {mc:.2f}\n"
            f"M máx. = {magnitud_maxima_ajuste:.2f}"
        )
        eje.text(
            0.97,
            0.56,
            texto_gr,
            transform=eje.transAxes,
            verticalalignment="center",
            horizontalalignment="right",
            bbox=dict(facecolor="white", alpha=0.82, edgecolor="gray"),
        )

        eje.legend(loc="upper right")
        figura.tight_layout()

        guardar_figura(
            figura,
            carpeta_salida,
            "06_Ley_Gutenberg_Richter.png",
        )

        mostrar_figura(figura, "Se muestra la figura final de Gutenberg-Richter. Revísela y cierre la ventana para decidir si acepta Mc.")
        plt.close(figura)

        aceptar = pedir_si_no(
            "¿Acepta Mc y la magnitud máxima del ajuste?",
            "s",
        )

        if aceptar == "s":
            break

        mc = pedir_numero("Nueva magnitud de corte Mc", mc)
        magnitud_maxima_ajuste = pedir_numero(
            "Nueva magnitud máxima utilizada en el ajuste lineal",
            magnitud_maxima_ajuste,
        )

    tabla_gr = pd.DataFrame(
        {
            "Magnitud": magnitudes_gr,
            "Numero_acumulado": numero_acumulado,
        }
    )
    tabla_gr.to_csv(carpeta_salida / "Tabla_Gutenberg_Richter.csv", index=False)

    print("\nRESULTADOS GUTENBERG-RICHTER")
    print("Mc = %.3f" % mc)
    print("Magnitud máxima del ajuste = %.3f" % magnitud_maxima_ajuste)
    print("a  = %.4f" % a)
    print("b  = %.4f" % b)

    pausa("Continuar...")

    # -----------------------------------------------------------------
    # PASO 6. LEY DE OMORI
    # -----------------------------------------------------------------
    print("\n" + "-" * 78)
    print("PASO 6. LEY DE OMORI")
    print("-" * 78)

    eventos_mayores = (
        region.sort_values("Magnitud", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print("\nDIEZ EVENTOS DE MAYOR MAGNITUD")
    for i in range(len(eventos_mayores)):
        fila = eventos_mayores.loc[i]
        print(
            f"{i + 1:2d}. {fila['Fecha']} · Mw {fila['Magnitud']:.2f} · "
            f"Lat {fila['Latitud']:.3f} · Lon {fila['Longitud']:.3f} · "
            f"Prof {fila['Profundidad']:.1f} km"
        )

    while True:
        numero_evento = pedir_entero("Número del evento principal", 1)

        if 1 <= numero_evento <= len(eventos_mayores):
            break

        print("El número está fuera del rango de la lista.")

    evento_principal = eventos_mayores.loc[numero_evento - 1]
    fecha_principal = evento_principal["Fecha"]

    dias_analisis = pedir_numero(
        "Duración analizada después del evento principal [días]", 30
    )
    ancho_bin = pedir_numero("Ancho del intervalo temporal [días]", 1)

    if dias_analisis <= 0 or ancho_bin <= 0:
        raise ValueError("La duración y el intervalo deben ser mayores que cero.")

    dias_antes = 10.0
    fecha_inicial_omori = fecha_principal - pd.to_timedelta(dias_antes, unit="D")
    fecha_final_omori = fecha_principal + pd.to_timedelta(dias_analisis, unit="D")

    # Ventana completa mostrada en la figura: 10 días antes y el periodo
    # posterior definido por el usuario.
    ventana_omori = region[
        (region["Fecha"] >= fecha_inicial_omori)
        & (region["Fecha"] <= fecha_final_omori)
    ].copy()

    ventana_omori["Tiempo_dias"] = (
        ventana_omori["Fecha"] - fecha_principal
    ).dt.total_seconds() / (24 * 3600)

    # La secuencia posterior se mantiene separada para guardar los eventos
    # posteriores al sismo principal.
    secuencia = ventana_omori[
        ventana_omori["Tiempo_dias"] > 0
    ].copy()

    if len(secuencia) < 3:
        raise ValueError(
            "La ventana elegida contiene muy pocos eventos posteriores "
            "al evento principal."
        )

    bordes_tiempo = np.arange(
        -dias_antes,
        dias_analisis + ancho_bin,
        ancho_bin,
    )

    if len(bordes_tiempo) < 3:
        raise ValueError("Se necesitan al menos dos intervalos temporales.")

    conteo, bordes_tiempo = np.histogram(
        ventana_omori["Tiempo_dias"],
        bins=bordes_tiempo,
    )

    centros_tiempo = 0.5 * (bordes_tiempo[:-1] + bordes_tiempo[1:])
    tasa = conteo / ancho_bin

    # El ajuste matemático comienza desde el día siguiente al evento
    # principal. Los intervalos previos y el primer día se muestran, pero no
    # intervienen en el cálculo de K, c y p.
    mascara_omori = (bordes_tiempo[:-1] >= 1.0) & (tasa > 0)
    x_omori = centros_tiempo[mascara_omori]
    y_omori = tasa[mascara_omori]

    if len(x_omori) < 3:
        raise ValueError(
            "No hay suficientes intervalos con eventos para ajustar Omori."
        )

    parametros, _ = curve_fit(
        omori,
        x_omori,
        y_omori,
        p0=[max(y_omori), 0.5, 1.0],
        bounds=([0.0, 0.01, 0.0], [np.inf, np.inf, np.inf]),
        maxfev=20000,
    )

    k, c, p = parametros
    tiempo_modelo = np.linspace(max(1.0, x_omori.min()), x_omori.max(), 300)

    figura, eje = plt.subplots(figsize=(10, 5.5))

    colores_barras = [
        "gray" if tiempo < 0 else "darkred"
        for tiempo in centros_tiempo
    ]

    eje.bar(
        centros_tiempo,
        tasa,
        width=0.90 * ancho_bin,
        color=colores_barras,
        edgecolor="black",
        alpha=0.80,
        label="Sismicidad por unidad de tiempo",
    )

    eje.plot(
        tiempo_modelo,
        omori(tiempo_modelo, k, c, p),
        "k--",
        linewidth=4,
        label="Ajuste Ley de Omori",
    )

    eje.axvline(
        0,
        color="navy",
        linestyle=":",
        linewidth=2.2,
        label="Evento principal (t = 0)",
    )

    eje.axvline(
        1,
        color="darkgreen",
        linestyle=":",
        linewidth=2.0,
        label="Inicio del ajuste (t = 1 día)",
    )

    eje.set_xlim(-dias_antes, dias_analisis)
    eje.set_xlabel("Días respecto del evento principal")
    eje.set_ylabel("Número de eventos por día")
    eje.set_title("Ajuste de la Ley de Omori")

    texto = (
        "Ley de Omori modificada\n"
        "n(t) = K / (t + c)^p\n"
        f"K = {k:.2f}\n"
        f"c = {c:.4f}\n"
        f"p = {p:.2f}"
    )
    eje.text(
        0.95,
        0.72,
        texto,
        transform=eje.transAxes,
        verticalalignment="center",
        horizontalalignment="right",
        bbox=dict(facecolor="white", alpha=0.75, edgecolor="gray"),
    )

    eje.legend(loc="center right")
    eje.grid(alpha=0.20, linestyle=":", axis="y")
    figura.tight_layout()

    guardar_figura(figura, carpeta_salida, "07_Ajuste_Ley_de_Omori.png")
    mostrar_figura(figura, "Se muestra la figura de la Ley de Omori. Revísela y cierre la ventana para continuar.")
    plt.close(figura)

    tabla_omori = pd.DataFrame(
        {
            "Tiempo_centro_dias": centros_tiempo,
            "Conteo": conteo,
            "Tasa_eventos_por_dia": tasa,
        }
    )
    tabla_omori.to_csv(carpeta_salida / "Tabla_Omori.csv", index=False)
    secuencia.to_csv(carpeta_salida / "Catalogo_secuencia_Omori.csv", index=False)

    # -----------------------------------------------------------------
    # RESUMEN FINAL
    # -----------------------------------------------------------------
    with (carpeta_salida / "Resumen_parametros.txt").open(
        "w", encoding="utf-8"
    ) as archivo:
        archivo.write("ACTIVIDAD 1 - RESUMEN DE PARÁMETROS\n")
        archivo.write("=" * 60 + "\n\n")

        archivo.write("REGIÓN ANALIZADA\n")
        archivo.write(f"Longitud: {lon_min:.4f} a {lon_max:.4f}°\n")
        archivo.write(f"Latitud: {lat_min:.4f} a {lat_max:.4f}°\n")
        archivo.write(f"Eventos: {len(region)}\n\n")

        archivo.write("SECCIÓN A-B\n")
        archivo.write(f"A = ({lon_a:.5f}, {lat_a:.5f})\n")
        archivo.write(f"B = ({lon_b:.5f}, {lat_b:.5f})\n")
        archivo.write(f"Longitud de A-B = {longitud_ab:.2f} km\n")
        archivo.write(f"Corredor = ±{semiancho:.3f}°\n")
        archivo.write(f"Eventos del perfil = {len(perfil)}\n\n")

        archivo.write("GUTENBERG-RICHTER\n")
        archivo.write(f"ΔM = {delta_m:.3f}\n")
        archivo.write(f"Mc = {mc:.3f}\n")
        archivo.write(
            f"Magnitud máxima del ajuste = {magnitud_maxima_ajuste:.3f}\n"
        )
        archivo.write(f"a = {a:.5f}\n")
        archivo.write(f"b = {b:.5f}\n\n")

        archivo.write("OMORI\n")
        archivo.write(f"Evento principal = {fecha_principal}\n")
        archivo.write(f"Magnitud principal = {evento_principal['Magnitud']:.3f}\n")
        archivo.write(f"Días mostrados antes del evento = {dias_antes:.3f}\n")
        archivo.write(f"Duración posterior = {dias_analisis:.3f} días\n")
        archivo.write("Inicio del ajuste matemático = 1.000 día\n")
        archivo.write(f"Intervalo = {ancho_bin:.3f} días\n")
        archivo.write(f"K = {k:.5f}\n")
        archivo.write(f"c = {c:.5f}\n")
        archivo.write(f"p = {p:.5f}\n")

    print("\n" + "=" * 78)
    print("TRABAJO FINALIZADO")
    print("=" * 78)
    print("Los resultados se guardaron en:")
    print(carpeta_salida.resolve())


In [2]:
main()

Backend gráfico activado: qtagg

ACTIVIDAD 1: CONTEXTO SISMOTECTÓNICO Y LEYES EMPÍRICAS
PROGRAMA DOCENTE BASADO EN EL NOTEBOOK ORIGINAL

RESUMEN DEL CATÁLOGO
Número de eventos: 2563
Fecha inicial: 2007-01-01 02:33:09.600000
Fecha final: 2017-01-01 10:20:10.700000
Latitud: -33.000 a -29.007°
Longitud: -72.981 a -69.012°
Profundidad: 0.0 a 60.0 km
Magnitud: 2.50 a 8.30

------------------------------------------------------------------------------
PASO 1. SISMICIDAD EN PLANTA
------------------------------------------------------------------------------
Figura guardada: 01_Sismicidad_general_en_planta.png

Se muestra la figura general de sismicidad en planta. Revísela y cierre la ventana para continuar.

------------------------------------------------------------------------------
PASO 2. DEFINIR LOS LÍMITES DEL ÁREA DE ANÁLISIS
------------------------------------------------------------------------------
Eventos incluidos en la región: 2563
Figura guardada: 02_Region_seleccionada.png


KeyboardInterrupt: Interrupted by user